# UrbanFlow — Core Analysis Walkthrough

Notebook này **không thay thế pipeline chính** và không phải một “tiểu luận thứ hai”.

Mục tiêu là minh họa ngắn gọn các bước cốt lõi đã có trong project: đọc dữ liệu đã xử lý, kiểm tra chất lượng, tái sử dụng logic làm sạch, tính KPI và tạo một số biểu đồ đại diện cho Overview, Temporal Patterns, Mobility Flow và Weather Impact.

Dữ liệu tham chiếu: **Citi Bike NYC — August 2026**.

## 1. Thiết lập môi trường và đường dẫn

Notebook được thiết kế để đặt trong thư mục `notebooks/` và chạy bằng **Run All**.

In [1]:
from pathlib import Path
import sys

import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

TRIPS_PATH = PROJECT_ROOT / "data" / "processed" / "trips" / "trips_2026_08.parquet"
WEATHER_PATH = PROJECT_ROOT / "data" / "processed" / "weather" / "weather_2026_08.parquet"
DATABASE_PATH = PROJECT_ROOT / "data" / "analytics" / "urbanflow.duckdb"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "citibike" / "2026-08"

for path in [TRIPS_PATH, WEATHER_PATH, DATABASE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required project file: {path}")

print("Project root :", PROJECT_ROOT)
print("Trips        :", TRIPS_PATH)
print("Weather      :", WEATHER_PATH)
print("DuckDB       :", DATABASE_PATH)

Project root : c:\Users\Admin\urbanflow
Trips        : c:\Users\Admin\urbanflow\data\processed\trips\trips_2026_08.parquet
Weather      : c:\Users\Admin\urbanflow\data\processed\weather\weather_2026_08.parquet
DuckDB       : c:\Users\Admin\urbanflow\data\analytics\urbanflow.duckdb


## 2. Đọc dữ liệu đã xử lý

UrbanFlow lưu dữ liệu chuyến đi đã làm sạch ở định dạng **Parquet**. Ta dùng `scan_parquet()` để giữ cách xử lý lazy của Polars.

In [2]:
trips = pl.scan_parquet(TRIPS_PATH)

row_count = trips.select(pl.len().alias("rows")).collect().item()
print(f"Rows: {row_count:,}")

print("\nSchema:")
print(trips.collect_schema())

print("\nSample:")
trips.head(5).collect()

Rows: 5,244,782

Schema:
Schema({'ride_id': String, 'rideable_type': String, 'started_at': Datetime(time_unit='us', time_zone=None), 'ended_at': Datetime(time_unit='us', time_zone=None), 'start_station_name': String, 'start_station_id': String, 'end_station_name': String, 'end_station_id': String, 'start_lat': Float64, 'start_lng': Float64, 'end_lat': Float64, 'end_lng': Float64, 'member_casual': String, 'duration_minutes': Float64, 'date': Date, 'hour': Int8, 'weekday': Int8, 'day': Int8, 'month': Int8})

Sample:


ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_minutes,date,hour,weekday,day,month
str,str,datetime[μs],datetime[μs],str,str,str,str,f64,f64,f64,f64,str,f64,date,i8,i8,i8,i8
"""E4A8132DACF670C3""","""electric_bike""",2026-08-10 14:31:22.100,2026-08-10 14:37:37.034,"""E 33 St & 5 Ave""","""6322.01""","""W 24 St & 7 Ave""","""6257.03""",40.747659,-73.984907,40.744876,-73.995299,"""member""",6.233333,2026-08-10,14,1,10,8
"""8382768C05C30BB4""","""electric_bike""",2026-08-10 08:04:28.348,2026-08-10 08:33:27.680,"""Emerson Pl & Myrtle Ave""","""4683.02""","""W 24 St & 7 Ave""","""6257.03""",40.693631,-73.962236,40.744876,-73.995299,"""member""",28.983333,2026-08-10,8,1,10,8
"""3A1C43832BBE4FCD""","""classic_bike""",2026-08-08 19:36:32.350,2026-08-08 19:43:47.890,"""Broadway & E 19 St""","""5980.11""","""W 24 St & 7 Ave""","""6257.03""",40.73829,-73.99006,40.744876,-73.995299,"""member""",7.25,2026-08-08,19,6,8,8
"""5C9BC6DFC4DEA183""","""electric_bike""",2026-08-05 14:34:27.749,2026-08-05 14:46:23.526,"""Sterling Pl & Bedford Ave""","""3993.03""","""Cumberland St & Lafayette Ave""","""4428.02""",40.672695,-73.954131,40.687534,-73.972652,"""member""",11.916667,2026-08-05,14,3,5,8
"""355579CC64D1BE20""","""electric_bike""",2026-08-01 14:16:44.531,2026-08-01 14:29:39.539,"""Sterling Pl & Bedford Ave""","""3993.03""","""Cumberland St & Lafayette Ave""","""4428.02""",40.672695,-73.954131,40.687534,-73.972652,"""member""",12.916667,2026-08-01,14,6,1,8


## 3. Kiểm tra chất lượng và minh họa bước làm sạch

Pipeline chính đã loại bản ghi thiếu `ride_id`/timestamp, loại chuyến có thời lượng không dương, giới hạn đúng kỳ phân tích và tạo các trường `duration_minutes`, `date`, `hour`, `weekday`, `day`, `month`.

In [4]:
quality = (
    trips
    .select(
        pl.len().alias("rows"),
        pl.col("ride_id").n_unique().alias("unique_ride_ids"),
        pl.col("duration_minutes").min().alias("min_duration"),
        pl.col("duration_minutes").median().alias("median_duration"),
        pl.col("duration_minutes").max().alias("max_duration"),
        pl.col("ride_id").null_count().alias("null_ride_id"),
        pl.col("started_at").null_count().alias("null_started_at"),
        pl.col("ended_at").null_count().alias("null_ended_at"),
    )
    .collect()
)
quality

rows,unique_ride_ids,min_duration,median_duration,max_duration,null_ride_id,null_started_at,null_ended_at
u32,u32,f64,f64,f64,u32,u32,u32
5244782,5244782,1.0,9.783333,1499.983333,0,0,0


In [5]:
from urbanflow.processing.clean_trips import clean_trips

raw_files = sorted(RAW_DIR.glob("*.csv"))

if raw_files:
    raw_sample = pl.scan_csv(
        raw_files[0],
        infer_schema_length=10_000,
    ).head(100_000)

    cleaned_sample = clean_trips(
        raw_sample,
        year=2026,
        month=8,
    ).collect()

    print("Raw sample file:", raw_files[0].name)
    print(f"Rows after cleaning sample: {cleaned_sample.height:,}")

    cleaned_sample.select(
        "ride_id",
        "started_at",
        "ended_at",
        "duration_minutes",
        "date",
        "hour",
        "weekday",
        "member_casual",
        "rideable_type",
    ).head(8)
else:
    print("Raw CSV files are not present. Skipping raw-cleaning demonstration.")

Raw CSV files are not present. Skipping raw-cleaning demonstration.


## 4. Tổng quan nhu cầu

Các KPI dưới đây được truy vấn từ bảng `trips` trong DuckDB — cùng lớp dữ liệu mà dashboard sử dụng.

In [17]:
con = duckdb.connect(str(DATABASE_PATH), read_only=True)

summary = con.sql(
    """
    SELECT
        COUNT(*) AS total_rides,
        MEDIAN(duration_minutes) AS median_duration,
        AVG(CASE WHEN member_casual = 'member' THEN 1 ELSE 0 END) * 100
            AS member_share,
        AVG(CASE WHEN rideable_type = 'electric_bike' THEN 1 ELSE 0 END) * 100
            AS electric_share
    FROM trips
    """
).pl()

summary

total_rides,median_duration,member_share,electric_share
i64,f64,f64,f64
5244782,9.783333,78.505456,73.053027


In [18]:
daily = con.sql(
    """
    SELECT date, rides
    FROM daily_metrics
    ORDER BY date
    """
).pl()

daily_fig = px.line(
    daily,
    x="date",
    y="rides",
    markers=True,
    title="Daily Ride Volume — August 2026",
    labels={"date": "Date", "rides": "Rides"},
)
daily_fig.update_traces(line_width=3)
daily_fig.update_layout(
    template="plotly_white",
    height=430,
    showlegend=False,
)
daily_fig.show()

**Nhận xét ngắn:** nhu cầu thay đổi rõ theo ngày, vì vậy chỉ nhìn tổng số chuyến của cả tháng sẽ che mất các biến động theo thời gian.

## 5. Temporal Patterns

Ta tính số chuyến trung bình theo từng `date × hour`, sau đó tổng hợp theo `weekday × hour` để mô tả nhịp nhu cầu điển hình.

In [19]:
temporal = con.sql(
    """
    WITH daily_hour AS (
        SELECT
            date,
            weekday,
            hour,
            COUNT(*) AS rides
        FROM trips
        GROUP BY date, weekday, hour
    )
    SELECT
        weekday,
        hour,
        AVG(rides) AS avg_rides
    FROM daily_hour
    GROUP BY weekday, hour
    ORDER BY weekday, hour
    """
).pl()

weekday_order = [1, 2, 3, 4, 5, 6, 7]

weekday_names = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

hours = list(range(24))

pivot = (
    temporal
    .pivot(
        values="avg_rides",
        index="weekday",
        on="hour",
        aggregate_function="first",
    )
    .sort("weekday")
)

z = pivot.select([str(h) for h in hours]).to_numpy()

heatmap_fig = go.Figure(
    data=go.Heatmap(
        z=z,
        x=hours,
        y=weekday_order,
        colorscale="Blues",
        colorbar={"title": "Avg rides"},
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Hour: %{x}:00<br>"
            "Average rides: %{z:,.0f}"
            "<extra></extra>"
        ),
    )
)

heatmap_fig.update_layout(
    title="Average Ride Demand by Weekday and Hour",
    xaxis_title="Hour of Day",
    yaxis_title=None,
    template="plotly_white",
    height=440,
)
heatmap_fig.update_yaxes(autorange="reversed")
heatmap_fig.show()

In [20]:
hourly_profile = con.sql(
    """
    WITH daily_hour AS (
        SELECT
            date,
            hour,
            COUNT(*) AS rides
        FROM trips
        GROUP BY date, hour
    )
    SELECT
        hour,
        AVG(rides) AS avg_rides
    FROM daily_hour
    GROUP BY hour
    ORDER BY hour
    """
).pl()

hour_fig = px.line(
    hourly_profile,
    x="hour",
    y="avg_rides",
    markers=True,
    title="Average Ride Demand by Hour",
    labels={
        "hour": "Hour of Day",
        "avg_rides": "Average Rides per Day",
    },
)
hour_fig.update_traces(line_width=3)
hour_fig.add_vrect(
    x0=7,
    x1=9,
    opacity=0.10,
    line_width=0,
    annotation_text="AM",
)
hour_fig.add_vrect(
    x0=16,
    x1=19,
    opacity=0.10,
    line_width=0,
    annotation_text="PM",
)
hour_fig.update_layout(
    template="plotly_white",
    height=420,
)
hour_fig.show()

**Nhận xét ngắn:** cấu trúc giờ cho thấy các đỉnh nhu cầu lặp lại rõ; trong snapshot tháng 08/2026, đỉnh trung bình nổi bật vào cuối buổi chiều, khoảng 17:00.

## 6. Mobility Flow

`net_flow = arrivals - departures`; `total_activity = arrivals + departures`; `imbalance_ratio = |net_flow| / total_activity`.

Notebook chỉ giữ một biểu đồ đại diện; bản đồ tương tác đầy đủ vẫn thuộc dashboard Streamlit.

In [21]:
outflow = con.sql(
    """
    SELECT
        station_name,
        net_flow
    FROM station_metrics
    WHERE total_activity >= 5000
    ORDER BY net_flow ASC
    LIMIT 7
    """
).pl()

inflow = con.sql(
    """
    SELECT
        station_name,
        net_flow
    FROM station_metrics
    WHERE total_activity >= 5000
    ORDER BY net_flow DESC
    LIMIT 7
    """
).pl()

station_flow = pl.concat([outflow, inflow]).with_columns(
    pl.when(pl.col("net_flow") >= 0)
      .then(pl.lit("Net inflow"))
      .otherwise(pl.lit("Net outflow"))
      .alias("flow_direction")
)

station_plot = station_flow.sort("net_flow")

flow_fig = px.bar(
    station_plot,
    x="net_flow",
    y="station_name",
    orientation="h",
    color="flow_direction",
    color_discrete_map={
        "Net inflow": "#0B6BFF",
        "Net outflow": "#FF5D6C",
    },
    title="Strongest Rider-driven Net Outflow and Inflow Stations",
    labels={
        "net_flow": "Net Flow (Arrivals − Departures)",
        "station_name": "Station",
        "flow_direction": "Direction",
    },
)

flow_fig.add_vline(
    x=0,
    line_width=1.5,
    line_dash="dash",
)

flow_fig.update_layout(
    template="plotly_white",
    height=540,
    width=950,
    yaxis_title=None,
    legend_title=None,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
    margin=dict(l=180, r=40, t=90, b=60),
)

flow_fig.show()

**Nhận xét ngắn:** net flow chỉ phản ánh dòng chuyến của người dùng, không bao gồm hoạt động điều chuyển xe của đơn vị vận hành; vì vậy nó được đọc như tín hiệu **rebalancing pressure**.

## 7. Weather Impact

UrbanFlow so sánh nhu cầu theo giờ với baseline `weekday × hour`.

`demand_vs_expected_pct = (rides / expected_rides - 1) × 100`

In [22]:
weather_summary = con.sql(
    """
    SELECT
        ROUND(
            CORR(temperature_c, demand_vs_expected_pct),
            3
        ) AS temp_correlation,
        ROUND(
            CORR(precipitation_mm, demand_vs_expected_pct),
            3
        ) AS precipitation_correlation,
        AVG(
            CASE
                WHEN precipitation_mm = 0
                THEN demand_vs_expected_pct
            END
        ) AS dry_adjusted,
        AVG(
            CASE
                WHEN precipitation_mm > 0
                THEN demand_vs_expected_pct
            END
        ) AS wet_adjusted
    FROM weather_ride_hourly
    """
).pl()

weather_summary

temp_correlation,precipitation_correlation,dry_adjusted,wet_adjusted
f64,f64,f64,f64
0.171,-0.427,4.173661,-14.532384


In [13]:
precip = con.sql(
    """
    SELECT
        precipitation_category,
        COUNT(*) AS hours,
        AVG(demand_vs_expected_pct) AS avg_vs_expected_pct,
        MEDIAN(demand_vs_expected_pct) AS median_vs_expected_pct
    FROM weather_ride_hourly
    GROUP BY precipitation_category
    """
).pl()

precipitation_order = [
    "No precipitation",
    "Light precipitation",
    "Moderate / heavy precipitation",
]

precip_colors = {
    "No precipitation": "#13C8A3",
    "Light precipitation": "#FF9F1C",
    "Moderate / heavy precipitation": "#FF5D6C",
}

precip_fig = px.bar(
    precip,
    x="precipitation_category",
    y="avg_vs_expected_pct",
    color="precipitation_category",
    category_orders={
        "precipitation_category": precipitation_order,
    },
    color_discrete_map=precip_colors,
    hover_data={
        "hours": True,
        "avg_vs_expected_pct": ":.1f",
    },
    title="Demand vs Expected by Precipitation",
    labels={
        "precipitation_category": "Weather",
        "avg_vs_expected_pct": "Demand vs Expected (%)",
        "hours": "Observed Hours",
    },
)

precip_fig.add_hline(
    y=0,
    line_dash="dash",
)
precip_fig.update_layout(
    template="plotly_white",
    height=430,
    showlegend=False,
)
precip_fig.show()

## 8. Kết luận ngắn

Các kết quả chính của snapshot tháng 08/2026:

- Dữ liệu sau xử lý gồm khoảng **5,24 triệu chuyến** và **744 giờ thời tiết**.
- Thành viên và xe điện chiếm tỷ trọng lớn trong tập dữ liệu.
- Nhu cầu có nhịp thời gian rõ ràng, với các đỉnh lặp lại theo giờ và khác biệt giữa ngày thường/cuối tuần.
- Một số trạm có net inflow hoặc net outflow nổi bật, tạo tín hiệu về áp lực tái cân bằng.
- Sau khi điều chỉnh theo baseline `weekday × hour`, lượng mưa có quan hệ âm với nhu cầu; đây là **mối liên hệ mô tả**, không phải kết luận nhân quả.

> Dashboard Streamlit vẫn là sản phẩm trực quan chính. Notebook này chỉ minh họa các bước phân tích cốt lõi có thể chạy lại.

In [14]:
con.close()
print("Notebook completed successfully.")

Notebook completed successfully.
